# Auto-Fix GitHub Pages Builds with Agenter

A common pain point: you push a docs change, the GitHub Pages build fails on some minor issue (broken link, YAML typo, config error), and you have to wait minutes for each retry.

**This notebook shows how to automate that fix-retry loop using Agenter.**

The workflow:
1. A docs build fails (MkDocs, Sphinx, Next.js, etc.)
2. Agenter reads the error log, understands the issue, and fixes the source files
3. The build is re-run to verify the fix
4. If it still fails, Agenter retries with the new error

At the end, we provide a **ready-to-use GitHub Actions workflow** that does this automatically on every push.

## Setup

```bash
pip install agenter
export ANTHROPIC_API_KEY="your-api-key"
```

In [1]:
import subprocess
import tempfile
from pathlib import Path

from agenter import AutonomousCodingAgent, CodingRequest, Verbosity, configure_logging
from agenter.data_models import Budget, CodingStatus

configure_logging(level="WARNING")

## 1. Simulate a Broken MkDocs Site

Let's create a small MkDocs project with a deliberate error — a nav entry pointing to a file that doesn't exist.

In [2]:
# Create a temporary MkDocs project with a broken config
workspace = tempfile.mkdtemp(prefix="agenter_pages_demo_")
docs_dir = Path(workspace) / "docs"
docs_dir.mkdir()

# mkdocs.yml references "guide.md" in the nav, but the file doesn't exist
(Path(workspace) / "mkdocs.yml").write_text("""\
site_name: My Project Docs
nav:
  - Home: index.md
  - Guide: guide.md
  - API Reference: api.md
strict: true
""")

(docs_dir / "index.md").write_text("""\
# Welcome to My Project

This is the homepage of the documentation.

See the [Guide](guide.md) for getting started.
""")

# Note: guide.md is MISSING — this will cause the build to fail
# api.md is also MISSING

print(f"Workspace: {workspace}")
print("Files created:")
for f in sorted(Path(workspace).rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(workspace)}")

Workspace: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_demo_n3in2yvr
Files created:
  docs/index.md
  mkdocs.yml


In [3]:
# Run the build — it should fail because guide.md and api.md don't exist
BUILD_CMD = "mkdocs build --strict"

result = subprocess.run(
    BUILD_CMD,
    shell=True,
    cwd=workspace,
    capture_output=True,
    text=True,
    timeout=120,
)

build_log = (result.stdout + "\n" + result.stderr).strip()
print(f"Exit code: {result.returncode}")
print(f"\nBuild output:\n{build_log}")

Exit code: 1

Build output:
Aborted with 3 warnings in strict mode!

INFO    -  Cleaning site directory
INFO    -  Building documentation to directory: /private/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_demo_n3in2yvr/site
WARNING -  A reference to 'guide.md' is included in the 'nav' configuration, which is not found in the documentation files.
WARNING -  A reference to 'api.md' is included in the 'nav' configuration, which is not found in the documentation files.
WARNING -  Doc file 'index.md' contains a link 'guide.md', but the target is not found among documentation files.


## 2. Auto-Fix with Agenter

Now we send the build error to Agenter and let it fix the docs.

In [4]:
FIX_PROMPT = """\
The documentation build failed. Fix the error so the build passes.

## Build command
```
{build_cmd}
```

## Build error output
```
{build_log}
```

## Instructions
- Read the files mentioned in the error to understand the issue.
- Apply the minimal fix needed to make the build pass.
- Do NOT change the meaning or content of existing documentation.
- Only fix syntax, references, config, or structural issues.
- Common issues: broken links, missing files in nav/toctree, YAML errors, bad indentation.
"""

agent = AutonomousCodingAgent(
    sandbox=False,  # Needs filesystem access to fix docs files
)

fix_result = await agent.execute(
    CodingRequest(
        prompt=FIX_PROMPT.format(build_cmd=BUILD_CMD, build_log=build_log),
        cwd=workspace,
        budget=Budget(max_cost_usd=2.0),
    ),
    verbosity=Verbosity.NORMAL,
)

print(f"\nStatus: {fix_result.status.value}")
print(f"Cost: ${fix_result.total_cost_usd:.4f}")
print(f"Files modified: {list(fix_result.files.keys())}")

╔════════════════════════════════════════════════════ Agenter ════════════════════════════════════════════════════╗
║ Working directory: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_demo_n3in2yvr                 ║
║ Model: us.anthropic.claude-sonnet-4-20250514-v1:0                                                               ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

▶ Iteration 1

╭─────────────────────────────────────────────────── 📝 Prompt ───────────────────────────────────────────────────╮
│ The documentation build failed. Fix the error so the build passes.                                              │
│                                                                                                                 │
│ ## Build command                                                                                                │
│ ```                                                                                                             │
│ mkdocs build --strict                                                                                           │
│ ```                                                                                                             │
│                                                                                                                 │
│ ## Build error output                                                                                           │
│ ```                                                                                                             │
│ Aborted with 3 warnings in strict mode!                                                                         │
│                                                                                                                 │
│ INFO    -  Cleaning site directory                                                                              │
│ INFO    -  Building documentation to directory:                                                                 │
│ /private/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_demo_n3in2yvr/site                       │
│ WARNING -  A reference to 'guide.md' is included in the 'nav' configuration, which is not found in the          │
│ documentation files.                                                                                            │
│ WARNING -  A referen                                                                                            │
│ ... (1,075 chars total)                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📨 Response (81 tokens) ────────────────────────────────────────────╮
│ I'll help you fix the documentation build errors. Let me start by examining the current directory structure and │
│ the configuration file to understand the issues.                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🔧 Tool ✓ ───────────────────────────────────────────────────╮
│ list_files .                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── 🔧 Tool ✓ ───────────────────────────────────────────────────╮
│ read_file docs/index.md                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 📨 Response (168 tokens) ────────────────────────────────────────────╮
│ Perfect! I've fixed all the issues that were causing the build to fail:                                         │
│                                                                                                                 │
│ 1. **Fixed navigation references**: Removed the references to missing `guide.md` and `api.md` files from the    │
│ `nav` section in `mkdocs.yml`                                                                                   │
│ 2. **Fixed broken link**: Removed the broken link to `guide.md` from the `index.md` file                        │
│                                                                                                                 │
│ The changes made were minimal and focused:                                                                      │
│ - Only removed the non-existent file references from navigation                                                 │
│ - Only removed the broken link from the content                                                                 │
│ - Preserved all existing content and meanin                                                                     │
│ ... (711 chars total)                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ✓ Validation ──────────────────────────────────────────────────╮
│ All checks passed                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╔════════════════════════════════════════════════════ Summary ════════════════════════════════════════════════════╗
║   Status:            ✅ Completed                                                                               ║
║   Iterations:        1                                                                                          ║
║   Tokens:            17,430                                                                                     ║
║   Duration:          28.5s                                                                                      ║
║   Files modified:    2                                                                                          ║
║                        • mkdocs.yml                                                                             ║
║                        • docs/index.md                                                                          ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝


Status: completed
Cost: $0.0000
Files modified: ['mkdocs.yml', 'docs/index.md']


## 3. Verify the Fix

Re-run the exact same build command to confirm it passes now.

In [5]:
# Re-run the build to verify the fix
verify = subprocess.run(
    BUILD_CMD,
    shell=True,
    cwd=workspace,
    capture_output=True,
    text=True,
    timeout=120,
)

if verify.returncode == 0:
    print("Build passed! The fix worked.")
else:
    print(f"Build still failing (exit code {verify.returncode}):")
    print(verify.stderr)

# Show what files exist now
print("\nFiles after fix:")
for f in sorted(Path(workspace).rglob("*")):
    if f.is_file() and "site" not in str(f):
        print(f"  {f.relative_to(workspace)}")

Build passed! The fix worked.

Files after fix:
  docs/index.md
  mkdocs.yml


## 4. The Full Fix Loop

In production, you want a retry loop: fix → rebuild → check → retry if still broken.

Here's a reusable function that does this:

In [6]:
async def autofix_build(
    build_cmd: str,
    cwd: str,
    max_retries: int = 3,
    max_cost_per_attempt: float = 2.0,
    backend: str = "anthropic-sdk",
) -> bool:
    """Run a build, and if it fails, use Agenter to fix it automatically.

    Args:
        build_cmd: The shell command to build docs (e.g. "mkdocs build --strict").
        cwd: Working directory for the build.
        max_retries: Max number of fix attempts.
        max_cost_per_attempt: Budget cap per fix attempt in USD.
        backend: Agenter backend ("anthropic-sdk", "claude-code", "codex", etc.)

    Returns:
        True if the build passes (either initially or after a fix).
    """
    # First, try the build as-is
    result = subprocess.run(
        build_cmd,
        shell=True,
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=300,
    )
    if result.returncode == 0:
        print("Build passed on first try!")
        return True

    build_log = (result.stdout + "\n" + result.stderr).strip()
    agent = AutonomousCodingAgent(backend=backend, sandbox=False)

    for attempt in range(1, max_retries + 1):
        print(f"\n--- Fix attempt {attempt}/{max_retries} ---")

        fix_result = await agent.execute(
            CodingRequest(
                prompt=FIX_PROMPT.format(build_cmd=build_cmd, build_log=build_log),
                cwd=cwd,
                budget=Budget(max_cost_usd=max_cost_per_attempt),
            ),
            verbosity=Verbosity.NORMAL,
        )

        if fix_result.status not in (CodingStatus.COMPLETED, CodingStatus.COMPLETED_WITH_LIMIT_EXCEEDED):
            print(f"Agent failed: {fix_result.status.value}")
            return False

        print(f"Agent fixed {len(fix_result.files)} file(s). Verifying build...")

        # Re-run the build
        result = subprocess.run(
            build_cmd,
            shell=True,
            cwd=cwd,
            capture_output=True,
            text=True,
            timeout=300,
        )
        if result.returncode == 0:
            print(f"Build passed after {attempt} fix(es)!")
            return True

        # Update error log for next attempt
        build_log = (result.stdout + "\n" + result.stderr).strip()
        print(f"Build still failing. New error:\n{build_log[:500]}")

    print(f"Could not fix the build after {max_retries} attempts.")
    return False

Let's test the loop with a fresh broken project — this time with multiple broken cross-references and a misnamed file in the nav:

In [7]:
# Create another broken MkDocs project — multiple broken cross-references
workspace_2 = tempfile.mkdtemp(prefix="agenter_pages_loop_")
docs_dir_2 = Path(workspace_2) / "docs"
docs_dir_2.mkdir()

# Nav references "getting-started.md" but the actual file is "getting_started.md" (underscore)
(Path(workspace_2) / "mkdocs.yml").write_text("""\
site_name: Loop Demo
nav:
  - Home: index.md
  - Getting Started: getting-started.md
  - FAQ: faq.md
strict: true
""")

(docs_dir_2 / "index.md").write_text("""\
# Loop Demo

Welcome! Check out the [Getting Started guide](getting-started.md) and the [FAQ](faq.md).
""")

# File exists but with wrong name (underscore instead of hyphen)
(docs_dir_2 / "getting_started.md").write_text("""\
# Getting Started

Follow these steps to get started.

For common questions, see the [FAQ](faq.md).
""")

# faq.md links to a non-existent "troubleshooting.md"
(docs_dir_2 / "faq.md").write_text("""\
# FAQ

**Q: Something isn't working?**
A: See the [Troubleshooting guide](troubleshooting.md).
""")

print("Files created:")
for f in sorted(Path(workspace_2).rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(workspace_2)}")

success = await autofix_build(
    build_cmd="mkdocs build --strict",
    cwd=workspace_2,
)

Files created:
  docs/faq.md
  docs/getting_started.md
  docs/index.md
  mkdocs.yml



--- Fix attempt 1/3 ---


╔════════════════════════════════════════════════════ Agenter ════════════════════════════════════════════════════╗
║ Working directory: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_loop_j4dyj6m9                 ║
║ Model: us.anthropic.claude-sonnet-4-20250514-v1:0                                                               ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

▶ Iteration 1

╭─────────────────────────────────────────────────── 📝 Prompt ───────────────────────────────────────────────────╮
│ The documentation build failed. Fix the error so the build passes.                                              │
│                                                                                                                 │
│ ## Build command                                                                                                │
│ ```                                                                                                             │
│ mkdocs build --strict                                                                                           │
│ ```                                                                                                             │
│                                                                                                                 │
│ ## Build error output                                                                                           │
│ ```                                                                                                             │
│ Aborted with 3 warnings in strict mode!                                                                         │
│                                                                                                                 │
│ INFO    -  Cleaning site directory                                                                              │
│ INFO    -  Building documentation to directory:                                                                 │
│ /private/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_loop_j4dyj6m9/site                       │
│ INFO    -  The following pages exist in the docs directory, but are not included in the "nav" configuration:    │
│   - getting_started.md                                                                                          │
│ WARNING -  A                                                                                                    │
│ ... (1,227 chars total)                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── 🔧 Tool ✓ ───────────────────────────────────────────────────╮
│ read_file docs/faq.md                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 📨 Response (150 tokens) ────────────────────────────────────────────╮
│ Perfect! I've fixed all the issues identified in the build error:                                               │
│                                                                                                                 │
│ 1. ✅ Fixed the navigation reference from `getting-started.md` to `getting_started.md` in `mkdocs.yml`          │
│ 2. ✅ Fixed the link in `index.md` from `getting-started.md` to `getting_started.md`                            │
│ 3. ✅ Removed the broken link to `troubleshooting.md` in `faq.md` and replaced it with appropriate generic text │
│                                                                                                                 │
│ The documentation build should now pass without warnings in strict mode. All files are properly referenced in   │
│ the navigation, and all i                                                                                       │
│ ... (538 chars total)                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ✓ Validation ──────────────────────────────────────────────────╮
│ All checks passed                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╔════════════════════════════════════════════════════ Summary ════════════════════════════════════════════════════╗
║   Status:            ✅ Completed                                                                               ║
║   Iterations:        1                                                                                          ║
║   Tokens:            28,492                                                                                     ║
║   Duration:          34.5s                                                                                      ║
║   Files modified:    3                                                                                          ║
║                        • mkdocs.yml                                                                             ║
║                        • docs/index.md                                                                          ║
║                        • docs/faq.md                                                                            ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

Agent fixed 3 file(s). Verifying build...
Build passed after 1 fix(es)!


## 5. GitHub Actions Workflow

Copy this workflow to `.github/workflows/pages-autofix.yml` in your repo.

It builds your docs, and if the build fails, uses Agenter to fix the issue and push the correction automatically.

```yaml
name: GitHub Pages with Auto-Fix

on:
  push:
    branches: [main]
    paths: ["docs/**", "mkdocs.yml"]  # Only trigger on docs changes

permissions:
  contents: write  # Needed to push fixes back

jobs:
  build-docs:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install docs dependencies
        run: pip install mkdocs mkdocs-material  # Adapt to your docs tool

      - name: Build docs
        id: build
        continue-on-error: true
        run: |
          mkdocs build --strict 2>&1 | tee /tmp/build-output.log
          echo "exit_code=${PIPESTATUS[0]}" >> "$GITHUB_OUTPUT"

      - name: Deploy to GitHub Pages
        if: steps.build.outputs.exit_code == '0'
        uses: peaceiris/actions-gh-pages@v4
        with:
          github_token: ${{ secrets.GITHUB_TOKEN }}
          publish_dir: ./site

      # --- Auto-fix starts here ---
      - name: Install agenter
        if: steps.build.outputs.exit_code != '0'
        run: pip install agenter

      - name: Auto-fix build errors
        if: steps.build.outputs.exit_code != '0'
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          python -c "
          import asyncio, subprocess
          from pathlib import Path
          from agenter import AutonomousCodingAgent, CodingRequest
          from agenter.data_models import Budget, CodingStatus

          FIX_PROMPT = '''
          The documentation build failed. Fix the error so the build passes.

          ## Build command
          mkdocs build --strict

          ## Build error output
          {error}

          ## Instructions
          - Read the files mentioned in the error to understand the issue.
          - Apply the minimal fix to make the build pass.
          - Do NOT change the meaning of existing documentation.
          '''

          async def main():
              build_log = Path('/tmp/build-output.log').read_text()
              agent = AutonomousCodingAgent(sandbox=False)

              for attempt in range(3):
                  result = await agent.execute(
                      CodingRequest(
                          prompt=FIX_PROMPT.format(error=build_log),
                          cwd='.',
                          budget=Budget(max_cost_usd=2.0),
                      )
                  )
                  if result.status not in (CodingStatus.COMPLETED, CodingStatus.COMPLETED_WITH_LIMIT_EXCEEDED):
                      raise SystemExit(f'Agent failed: {result.status.value}')

                  verify = subprocess.run('mkdocs build --strict', shell=True,
                      capture_output=True, text=True)
                  if verify.returncode == 0:
                      print(f'Fixed after {attempt + 1} attempt(s)')
                      return
                  build_log = verify.stdout + verify.stderr

              raise SystemExit('Could not fix build after 3 attempts')

          asyncio.run(main())
          "

      - name: Commit and push fix
        if: steps.build.outputs.exit_code != '0'
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add -A
          git diff --cached --quiet && echo "No changes to commit" && exit 0
          git commit -m "fix(docs): auto-fix build error via agenter"
          git push

      - name: Re-deploy after fix
        if: steps.build.outputs.exit_code != '0'
        run: mkdocs build --strict

      - name: Deploy fixed site to GitHub Pages
        if: steps.build.outputs.exit_code != '0'
        uses: peaceiris/actions-gh-pages@v4
        with:
          github_token: ${{ secrets.GITHUB_TOKEN }}
          publish_dir: ./site
```

### Setup instructions

1. Add `ANTHROPIC_API_KEY` to your repo secrets (Settings → Secrets → Actions)
2. Copy the workflow above to `.github/workflows/pages-autofix.yml`
3. Adapt the build command and docs dependencies to your project (Sphinx, Next.js, etc.)
4. Push and watch it work!

## Cleanup

In [8]:
import shutil

for ws in [workspace, workspace_2]:
    try:
        shutil.rmtree(ws)
        print(f"Cleaned up: {ws}")
    except Exception:
        pass

Cleaned up: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_demo_n3in2yvr


Cleaned up: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pages_loop_j4dyj6m9
